# 📗 Module 04 · Notebook 05 — SLM Spesialis: Kecil tapi Cakap

"Kenapa banyak Small Language Model (SLM) dirilis, padahal katanya bodoh?" Justru karena di dunia
nyata **SLM bukan generalis serba-bisa — mereka pekerja spesialis yang efisien.**

## Kenapa SLM bukan bodoh?
1. **"Kecil" itu relatif.** Qwen3-1.7B hari ini lebih pintar daripada model raksasa beberapa tahun lalu — berkat data & teknik, terutama **distillation** (model kecil belajar dari output model raksasa).
2. **Alat yang tepat.** Klasifikasi, ekstraksi, routing, redaksi PII, jawab-dari-konteks — **tak perlu 70B**.
3. **Spesialisasi > generalisasi.** SLM yang di-fine-tune untuk satu tugas bisa **mengalahkan model raksasa generalis** di tugas itu. ("Sempit tapi dalam.")
4. **Murah, cepat, privat.** 10-100× lebih murah/cepat; bisa jalan on-device (data tak keluar).
5. **Masa depan Agentic AI.** Riset NVIDIA: agen melakukan banyak sub-tugas sempit & berulang → ideal untuk SLM murah-cepat.

## Yang akan kita buktikan
Latih **Qwen3-1.7B** jadi spesialis **ekstraksi → JSON**, lalu tunjukkan ia **mengalahkan generalis
SmolLM3-3B** (hampir 2× lebih besar) di tugas itu — dengan angka, bukan klaim.

In [ ]:
# Qwen3 & SmolLM3 butuh transformers >=4.53; pin <5 (gotcha 4-bit & API lama)
!pip install -q "transformers>=4.53,<5" peft bitsandbytes accelerate datasets

In [ ]:
import torch, json, re, time, random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SPECIALIST = "Qwen/Qwen3-1.7B"           # akan kita fine-tune jadi spesialis
GENERALIST = "HuggingFaceTB/SmolLM3-3B"  # generalis pembanding (~2x lebih besar)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def gpu_mem(tag=""):
    if not torch.cuda.is_available(): return
    print(f"[{tag}] allocated={torch.cuda.memory_allocated()/1e9:.2f} GB | reserved={torch.cuda.memory_reserved()/1e9:.2f} GB")

## 1. Tugas: Ekstraksi Pesanan → JSON

Ubah teks pesanan Indonesia yang berantakan menjadi JSON terstruktur:
`{"produk":..., "jumlah":..., "harga_satuan":..., "total":...}`.

Dua tantangan sekaligus: **format JSON valid** + **hitung** `total = jumlah × harga_satuan`
(termasuk membaca angka gaya Indonesia seperti "25.000"). Tugas ini sengaja sulit untuk model
kecil zero-shot, tapi mudah dipelajari lewat fine-tune — itulah esensi "sempit tapi dalam".

In [ ]:
random.seed(42)
PRODUK = ["minyak goreng","beras","gula pasir","kopi bubuk","teh celup","sabun mandi",
          "mie instan","tepung terigu","susu kental","garam"]
SATUAN = ["botol","kg","bungkus","kotak","sachet","pak"]
HARGA  = [5000,8500,12000,14000,17500,25000,30000,45000]
TEMPLATES = [
    "Tolong kirim {j} {s} {p}, harga {hf} per {s}.",
    "Saya mau pesan {p} sebanyak {j} {s}, harganya {hf}.",
    "Order: {p} {j} {s}, @{hf}.",
    "Pesanan {j} {s} {p}, harga satuan {hf} rupiah.",
    "minta {p} {j} {s} ya, {hf} satunya",
]
def gen_one():
    p=random.choice(PRODUK); s=random.choice(SATUAN)
    j=random.randint(1,20); h=random.choice(HARGA)
    hf=f"{h:,}".replace(",",".")                 # 25000 -> "25.000" (gaya Indonesia)
    text=random.choice(TEMPLATES).format(p=p,s=s,j=j,hf=hf)
    gold={"produk":p,"jumlah":j,"harga_satuan":h,"total":j*h}
    return {"text":text,"gold":gold}

data=[gen_one() for _ in range(250)]
train_data, test_data = data[:200], data[200:]
print(f"train={len(train_data)} | test={len(test_data)}")
print("contoh:", train_data[0]["text"], "->", json.dumps(train_data[0]["gold"], ensure_ascii=False))

## 2. Baseline Zero-shot (Belum Dilatih)

Sebelum fine-tune, ukur dua baseline pada teks uji yang sama:
- **Qwen3-1.7B base** (kecil, belum dilatih tugas ini)
- **SmolLM3-3B** (generalis, ~2× lebih besar)

Metrik: **valid-JSON rate** (output bisa di-parse?) + **akurasi field** (produk/jumlah/harga/total benar?) + **kecepatan** (ms/contoh).

In [ ]:
SYS = ("Ekstrak detail pesanan menjadi JSON dengan key persis: produk, jumlah, harga_satuan, total. "
       "harga_satuan dan total berupa angka tanpa titik/koma. Keluarkan HANYA JSON, tanpa penjelasan.")

def extract(model, tok, text, max_new_tokens=64):
    messages=[{"role":"system","content":SYS},{"role":"user","content":text}]
    try:
        inputs=tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True, enable_thinking=False).to(model.device)
    except TypeError:
        inputs=tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True).to(model.device)
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse_json(s):
    m=re.search(r"\{.*\}", s, re.S)
    if not m: return None
    try: return json.loads(m.group())
    except Exception: return None

def evaluate(model, tok, dataset, label=""):
    valid=0; hits=0; total=0; t0=time.time()
    for ex in dataset:
        obj=parse_json(extract(model, tok, ex["text"])); g=ex["gold"]
        if obj is None:
            total+=len(g); continue
        valid+=1
        for k,v in g.items():
            total+=1
            if str(obj.get(k)).strip()==str(v).strip(): hits+=1
    ms=(time.time()-t0)/len(dataset)*1000
    res={"label":label,"valid":valid/len(dataset),"field_acc":hits/total,"ms":ms}
    print(f"  [{label}] valid-JSON={res['valid']:.0%} | field-acc={res['field_acc']:.0%} | {ms:.0f} ms/contoh")
    return res

def load_4bit(name):
    bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    tok=AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
    return mdl, tok

In [ ]:
# Baseline 1 — Qwen3-1.7B base (4-bit, nanti kita fine-tune model ini)
model, tok = load_4bit(SPECIALIST)
gpu_mem("Qwen3-1.7B base")
print("Contoh output base (zero-shot):")
print("  teks:", test_data[0]["text"])
print("  out :", extract(model, tok, test_data[0]["text"]))
r_base = evaluate(model, tok, test_data, "Qwen3-1.7B base")

## 3. Baseline Generalis SmolLM3-3B

Model generalis yang kuat & hampir 2× lebih besar. Kita muat 4-bit hanya untuk inferensi, ukur,
lalu bebaskan memorinya sebelum training.

In [ ]:
gen_model, gen_tok = load_4bit(GENERALIST)
gpu_mem("SmolLM3-3B")
r_gen = evaluate(gen_model, gen_tok, test_data, "SmolLM3-3B generalis")
del gen_model, gen_tok
torch.cuda.empty_cache()
gpu_mem("setelah SmolLM3 dibebaskan")

## 4. Latih Spesialis (QLoRA)

Sekarang kita ubah Qwen3-1.7B base menjadi **spesialis** lewat QLoRA: format data dengan chat
template (jawaban = JSON gold), lalu latih adapter kecil. Qwen3 itu **hybrid-thinking**; untuk
tugas format ini kita pakai `enable_thinking=False` agar output langsung JSON.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

def to_text(ex):
    messages=[{"role":"system","content":SYS},{"role":"user","content":ex["text"]},
              {"role":"assistant","content":json.dumps(ex["gold"], ensure_ascii=False)}]
    try:
        return {"text": tok.apply_chat_template(messages, tokenize=False, enable_thinking=False)}
    except TypeError:
        return {"text": tok.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(train_data).map(to_text)
def tokenize(ex): return tok(ex["text"], truncation=True, max_length=256)
tok_ds = train_ds.map(tokenize, remove_columns=train_ds.column_names)

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args=TrainingArguments(output_dir="qwen3-extractor-lora", per_device_train_batch_size=4,
    gradient_accumulation_steps=2, num_train_epochs=5, learning_rate=2e-4, fp16=True,
    logging_steps=10, optim="paged_adamw_8bit", report_to="none", save_strategy="no")
Trainer(model=model, args=args, train_dataset=tok_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)).train()
model.config.use_cache=True; model.eval()

## 5. Head-to-Head: Spesialis vs Generalis

Sekarang ukur spesialis 1.7B (fine-tuned) dan bandingkan dengan dua baseline. Hipotesis:
**spesialis kecil mengalahkan generalis yang 2× lebih besar** — lebih akurat, lebih cepat.

In [ ]:
r_spec = evaluate(model, tok, test_data, "Qwen3-1.7B SPESIALIS")

print("\n=== HEAD-TO-HEAD ===")
print(f"{'model':30}{'params':>8}{'valid-JSON':>12}{'field-acc':>11}{'ms/contoh':>11}")
for r,pp in [(r_base,"1.7B"),(r_gen,"3.0B"),(r_spec,"1.7B")]:
    print(f"{r['label']:30}{pp:>8}{r['valid']:>11.0%}{r['field_acc']:>11.0%}{r['ms']:>10.0f}")

print("\n--- contoh: base vs spesialis (model yang SAMA, beda adapter) ---")
ex=test_data[1]
with model.disable_adapter():
    base_out=extract(model, tok, ex["text"])
print("teks     :", ex["text"])
print("gold     :", json.dumps(ex["gold"], ensure_ascii=False))
print("base     :", base_out)
print("spesialis:", extract(model, tok, ex["text"]))

## Ringkasan & Jembatan

| Pelajaran | Inti |
|-----------|------|
| SLM bukan bodoh | mereka **spesialis efisien**, bukan generalis serba-bisa |
| Spesialis > generalis | SLM kecil yang di-fine-tune **mengalahkan model 2× lebih besar** di tugasnya |
| Murah & cepat | lebih sedikit parameter → ms/contoh lebih rendah → biaya produksi turun |
| Resep | dataset tugas → QLoRA (notebook 03) → ukur jujur (notebook 04) |

Di **notebook 06** kita pakai resep yang sama untuk **tugas spesialis lain** (klasifikasi/intent
routing, sentimen/aspect, asisten domain) + benchmark ukuran 0.6B vs 1.7B vs 3B.